# 69-claim neural CFR+ matched schedule screen

This tests four action-expansion schedules on the 69-claim `r6_s4_h4_hp2ptfhq_ss` game.

Each branch starts from the same existing neural CFR+ checkpoint, trains for 8 measured neural-training minutes, saves playable average-policy snapshots at 4 and 8 minutes, then runs a 2-minute action-conditioned fitted-return BR against each snapshot.

The goal is not resumability. The goal is a clean matched-rate screen:

- A: cap16 random
- B: full first traverser decision, then cap16 random
- C: full first traverser decision, then cap16 hash/block coverage
- D: full first two traverser decisions, then cap16 random

The traversal counts are reduced for heavier expansion variants to keep iteration rate in the same ballpark.

In [ ]:
from datetime import datetime, timezone
import gc
import json
from pathlib import Path
import re
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.serialization import save_policy
from liars_poker.training.br_runs import run_best_responder

assert torch.cuda.is_available(), 'This screen is intended for a CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('repo:', REPO_ROOT)
print('torch:', torch.__version__)
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
SPEC = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(SPEC).claims) == 69

# Leave as None to use the newest matching run with latest_checkpoint.pt.
SOURCE_RUN_DIR = None

TRAINING_MINUTES = 8.0
SNAPSHOT_MINUTES = (4.0, 8.0)
OUTPUT_ROOT = REPO_ROOT / 'artifacts' / 'cfr_plus_69_claim_matched_schedule_screens'

VARIANTS = [
    {
        'label': 'A cap16 random',
        'traversals_per_player': 256,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (16,),
        'sample_mode': 'random',
        'traversal_batch_size': 64,
    },
    {
        'label': 'B full-first cap16 random',
        'traversals_per_player': 128,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (69, 16),
        'sample_mode': 'random',
        'traversal_batch_size': 32,
    },
    {
        'label': 'C full-first cap16 hash',
        'traversals_per_player': 128,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (69, 16),
        'sample_mode': 'hash',
        'traversal_batch_size': 32,
    },
    {
        'label': 'D full-first-two cap16 random',
        'traversals_per_player': 64,
        'regret_steps': 24,
        'strategy_steps': 6,
        'schedule': (69, 69, 16),
        'sample_mode': 'random',
        'traversal_batch_size': 16,
    },
]

BR_MINUTES = 2.0
BR_SEED = 7
BR_EVALUATE_EVERY_MINUTES = 1.0
BR_EVAL_EPISODES_PER_ROLE = 100_000
BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024
BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': str(device),
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
}

pd.DataFrame(VARIANTS)

In [ ]:
def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return list(value)
    if hasattr(value, 'item'):
        return value.item()
    return str(value)


def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, default=json_default) + '\n')


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def slug(label):
    return re.sub(r'[^a-zA-Z0-9_]+', '_', label).strip('_').lower()


def find_source_run():
    short = SPEC.to_short_str()
    candidates = []
    for checkpoint in (REPO_ROOT / 'artifacts').rglob('latest_checkpoint.pt'):
        run_dir = checkpoint.parent
        if short not in run_dir.name:
            continue
        if not (run_dir / 'manifest.json').exists():
            continue
        candidates.append(run_dir)
    if not candidates:
        raise FileNotFoundError(f'No matching {short} run with latest_checkpoint.pt under artifacts/.')
    return max(candidates, key=lambda p: (p / 'latest_checkpoint.pt').stat().st_mtime)


def configure_variant(trainer, variant):
    trainer.traversal_backend = 'gpu_native'
    trainer.traversal_batch_size = int(variant['traversal_batch_size'])
    trainer.regret_train_steps = int(variant['regret_steps'])
    trainer.strategy_train_steps = int(variant['strategy_steps'])
    trainer.traverser_action_sample_count = None
    trainer.traverser_action_sample_fraction = None
    trainer.traverser_action_full_first = False
    trainer.traverser_action_sample_schedule = tuple(int(x) for x in variant['schedule'])
    trainer.traverser_action_priority_count = 0
    trainer.traverser_action_baseline = 'none'
    trainer.traverser_action_sample_mode = variant.get('sample_mode', 'random')
    trainer._gpu_traverser = None
    return trainer


def mean_float(values):
    values = list(values or [])
    return float(sum(values) / len(values)) if values else float('nan')


def flatten_training_record(label, variant, measured_s, iteration_wall_s, record):
    timing = record.get('timing', {}) or {}
    sampling = record.get('action_sampling', {}) or {}
    return {
        'configuration': label,
        'iteration': int(record['iteration']),
        'branch_iteration': int(record.get('branch_iteration', record['iteration'])),
        'measured_training_s': float(measured_s),
        'measured_training_min': float(measured_s) / 60.0,
        'iteration_wall_s': float(iteration_wall_s),
        'traversals_per_player': int(variant['traversals_per_player']),
        'regret_steps': int(variant['regret_steps']),
        'strategy_steps': int(variant['strategy_steps']),
        'schedule': list(variant['schedule']),
        'sample_mode': variant.get('sample_mode', 'random'),
        'traversal_batch_size': int(variant['traversal_batch_size']),
        'regret_loss': mean_float(record.get('regret_loss')),
        'strategy_loss': mean_float(record.get('strategy_loss')),
        'new_regret_records': int(sum(record.get('new_regret_records', []) or [])),
        'new_strategy_records': int(sum(record.get('new_strategy_records', []) or [])),
        'traversal_s': float(timing.get('traversal_s', float('nan'))),
        'regret_training_s': float(timing.get('regret_training_s', float('nan'))),
        'strategy_training_s': float(timing.get('strategy_training_s', float('nan'))),
        'claim_edge_fraction': float(sampling.get('claim_edge_fraction', float('nan'))),
        'regret_weight_ess_fraction': float(sampling.get('regret_weight_ess_fraction', float('nan'))),
        'max_regret_weight': float(sampling.get('max_regret_weight', float('nan'))),
    }


def save_average_snapshot(trainer, branch_dir, target_min, measured_s):
    policy_dir = branch_dir / 'snapshots' / f'target_{int(round(target_min)):02d}m_iter_{trainer.iteration:08d}'
    save_policy(trainer.average_policy(), str(policy_dir))
    return {
        'target_min': float(target_min),
        'snapshot_training_min': float(measured_s) / 60.0,
        'snapshot_iteration': int(trainer.iteration),
        'policy_dir': str(policy_dir),
    }


SOURCE_RUN_DIR = Path(SOURCE_RUN_DIR) if SOURCE_RUN_DIR is not None else find_source_run()
CHECKPOINT_PATH = SOURCE_RUN_DIR / 'latest_checkpoint.pt'
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

run_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_ROOT = OUTPUT_ROOT / f'{SPEC.to_short_str()}___{run_id}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
manifest = {
    'run_type': 'cfr_plus_69_claim_matched_schedule_screen',
    'spec': SPEC.to_json(),
    'source_run_dir': str(SOURCE_RUN_DIR),
    'source_checkpoint': str(CHECKPOINT_PATH),
    'training_minutes': TRAINING_MINUTES,
    'snapshot_minutes': SNAPSHOT_MINUTES,
    'br_minutes': BR_MINUTES,
    'variants': VARIANTS,
}
(RUN_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')
print('source:', SOURCE_RUN_DIR)
print('run root:', RUN_ROOT)

In [ ]:
training_summaries = []
budget_s = 60.0 * TRAINING_MINUTES
snapshot_targets_s = [60.0 * minute for minute in SNAPSHOT_MINUTES]

for variant in VARIANTS:
    label = variant['label']
    branch_dir = RUN_ROOT / slug(label)
    branch_dir.mkdir(parents=True, exist_ok=True)
    training_path = branch_dir / 'training.jsonl'

    print('\n===', label, '===')
    trainer = DeepCFRPlusTrainer.load_checkpoint(CHECKPOINT_PATH, device=device)
    trainer = configure_variant(trainer, variant)
    start_iteration = int(trainer.iteration)

    measured_s = 0.0
    snapshots = []
    remaining_snapshot_targets = list(snapshot_targets_s)
    last_print_min = -1

    while measured_s < budget_s:
        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=int(variant['traversals_per_player']))
        iteration_s = time.perf_counter() - start
        measured_s += iteration_s

        record['branch_iteration'] = int(trainer.iteration) - start_iteration
        flat = flatten_training_record(label, variant, measured_s, iteration_s, record)
        append_jsonl(training_path, flat)

        while remaining_snapshot_targets and measured_s >= remaining_snapshot_targets[0]:
            target_s = remaining_snapshot_targets.pop(0)
            snap = save_average_snapshot(trainer, branch_dir, target_s / 60.0, measured_s)
            snap['snapshot_branch_iteration'] = int(trainer.iteration) - start_iteration
            snapshots.append(snap)
            append_jsonl(branch_dir / 'snapshots.jsonl', snap)
            print(
                f"snapshot {label}: target={snap['target_min']:.0f}m "
                f"actual={snap['snapshot_training_min']:.2f}m iter={trainer.iteration} "
                f"branch_iter={snap['snapshot_branch_iteration']}"
            )

        current_min = int(measured_s // 60)
        if current_min > last_print_min:
            last_print_min = current_min
            print(
                f"{label}: train={measured_s / 60.0:.1f}m "
                f"branch_iter={int(trainer.iteration) - start_iteration} iter_s={iteration_s:.3f} "
                f"edges={flat['claim_edge_fraction']:.3f} ess={flat['regret_weight_ess_fraction']:.3f}"
            )

    if len(snapshots) < len(SNAPSHOT_MINUTES):
        existing_targets = {snap['target_min'] for snap in snapshots}
        for target_min in SNAPSHOT_MINUTES:
            if target_min not in existing_targets:
                snap = save_average_snapshot(trainer, branch_dir, target_min, measured_s)
                snap['snapshot_branch_iteration'] = int(trainer.iteration) - start_iteration
                snapshots.append(snap)
                append_jsonl(branch_dir / 'snapshots.jsonl', snap)

    records = read_jsonl(training_path)
    summary = {
        'configuration': label,
        'branch_dir': str(branch_dir),
        'final_policy_dir': str(Path(snapshots[-1]['policy_dir'])),
        'snapshots': snapshots,
        'measured_training_min': float(measured_s) / 60.0,
        'start_iteration': int(start_iteration),
        'end_iteration': int(trainer.iteration),
        'branch_iterations_completed': int(trainer.iteration) - start_iteration,
        'iterations_per_min': (int(trainer.iteration) - start_iteration) / (float(measured_s) / 60.0),
        'mean_traversal_s': mean_float(row['traversal_s'] for row in records),
        'mean_regret_fit_s': mean_float(row['regret_training_s'] for row in records),
        'mean_strategy_fit_s': mean_float(row['strategy_training_s'] for row in records),
        'mean_claim_edge_fraction': mean_float(row['claim_edge_fraction'] for row in records),
        'mean_regret_weight_ess_fraction': mean_float(row['regret_weight_ess_fraction'] for row in records),
        'largest_regret_weight': max(row['max_regret_weight'] for row in records),
        'variant': variant,
    }
    (branch_dir / 'summary.json').write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')
    training_summaries.append(summary)

    del trainer
    gc.collect()
    torch.cuda.empty_cache()

summary_df = pd.DataFrame(training_summaries).set_index('configuration')
summary_df[[
    'measured_training_min', 'branch_iterations_completed', 'iterations_per_min',
    'mean_traversal_s', 'mean_regret_fit_s', 'mean_strategy_fit_s',
    'mean_claim_edge_fraction', 'mean_regret_weight_ess_fraction', 'largest_regret_weight',
]]

In [ ]:
br_rows = []

for summary in training_summaries:
    label = summary['configuration']
    branch_dir = Path(summary['branch_dir'])
    for snapshot in summary['snapshots']:
        target_min = snapshot['target_min']
        policy_dir = Path(snapshot['policy_dir'])
        br_dir = branch_dir / 'br_2m_action_conditioned' / f'target_{int(round(target_min)):02d}m' / f'seed_{BR_SEED}'
        print(f"\n=== BR {label} target={target_min:.0f}m ===")
        result = run_best_responder(
            policy_dir,
            method='action_conditioned_fitted_return',
            minutes=BR_MINUTES,
            trainer_kwargs={**BR_TRAINER_KWARGS, 'seed': BR_SEED},
            episodes_per_role=BR_EPISODES_PER_ROLE,
            rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
            evaluate_every_minutes=BR_EVALUATE_EVERY_MINUTES,
            eval_episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
            exact_evaluation=False,
            run_dir=br_dir,
            debug=False,
        )
        evals = result.evaluation_records
        final_eval = evals[-1]
        best_estimate = max(ev['exploitability_estimate'] for ev in evals)
        best_lower = max(ev['exploitability_lower_bound'] for ev in evals)
        row = {
            'configuration': label,
            'snapshot_target_min': float(target_min),
            'snapshot_training_min': float(snapshot['snapshot_training_min']),
            'snapshot_iteration': int(snapshot['snapshot_iteration']),
            'snapshot_branch_iteration': int(snapshot.get('snapshot_branch_iteration', snapshot['snapshot_iteration'])),
            'responder_iterations': int(final_eval['iteration']),
            'responder_training_min': float(result.measured_training_s) / 60.0,
            'final_p_first': float(final_eval['p_first']),
            'final_p_second': float(final_eval['p_second']),
            'final_discovered_estimate': float(final_eval['exploitability_estimate']),
            'final_conservative_lower_bound': float(final_eval['exploitability_lower_bound']),
            'best_discovered_estimate': float(best_estimate),
            'best_conservative_lower_bound': float(best_lower),
            'br_run_dir': str(result.run_dir),
        }
        append_jsonl(RUN_ROOT / 'br_summary.jsonl', row)
        br_rows.append(row)
        print(
            f"best={best_estimate:.6f} lcb={best_lower:.6f} "
            f"p_first={final_eval['p_first']:.4f} p_second={final_eval['p_second']:.4f}"
        )

        del result
        gc.collect()
        torch.cuda.empty_cache()

br_df = pd.DataFrame(br_rows)
br_df.sort_values(['snapshot_target_min', 'best_discovered_estimate'])

In [ ]:
training_rows = []
for summary in training_summaries:
    training_rows.extend(read_jsonl(Path(summary['branch_dir']) / 'training.jsonl'))
training_df = pd.DataFrame(training_rows)
br_df = pd.DataFrame(read_jsonl(RUN_ROOT / 'br_summary.jsonl'))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for label, group in training_df.groupby('configuration'):
    axes[0, 0].plot(group['measured_training_min'], group['branch_iteration'], label=label)
    axes[0, 1].plot(group['measured_training_min'], group['traversal_s'], label=label)
    axes[0, 2].plot(group['measured_training_min'], group['claim_edge_fraction'], label=label)
    axes[1, 0].plot(group['measured_training_min'], group['regret_weight_ess_fraction'], label=label)
    axes[1, 1].plot(group['measured_training_min'], group['max_regret_weight'], label=label)
    axes[1, 2].plot(group['measured_training_min'], group['new_regret_records'], label=label)

axes[0, 0].set_title('Iterations completed')
axes[0, 1].set_title('Traversal seconds per iteration')
axes[0, 2].set_title('Claim-edge fraction')
axes[1, 0].set_title('Regret-weight ESS fraction')
axes[1, 1].set_title('Largest inverse-path regret weight')
axes[1, 1].set_yscale('log')
axes[1, 2].set_title('New regret records per iteration')
for ax in axes.flat:
    ax.set_xlabel('Measured CFR+ branch-training minutes')
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, group in br_df.groupby('configuration'):
    group = group.sort_values('snapshot_target_min')
    axes[0].plot(group['snapshot_target_min'], group['best_discovered_estimate'], marker='o', label=label)
    axes[1].plot(group['snapshot_target_min'], group['best_conservative_lower_bound'], marker='o', label=label)
axes[0].set_title('2-minute BR best discovered estimate')
axes[1].set_title('2-minute BR conservative lower bound')
for ax in axes:
    ax.set_xlabel('CFR+ branch snapshot target minute')
    ax.set_ylabel('Discovered exploitability')
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

final_br = br_df.sort_values(['snapshot_target_min', 'best_discovered_estimate'])
final_br[[
    'configuration', 'snapshot_target_min', 'snapshot_iteration', 'responder_training_min',
    'best_discovered_estimate', 'best_conservative_lower_bound', 'final_p_first', 'final_p_second',
]]